# Customer Churn Analysis



In [1]:
# 1) Install libraries (run once)
%pip install pandas sqlalchemy pymysql cryptography

   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 932.9 kB/s eta 0:00:04
   ---------- ----------------------------- 1.0/3.8 MB 1.3 MB/s eta 0:00:03
   ------------- -------------------------- 1.3/3.8 MB 1.4 MB/s eta 0:00:02
   ------------------- -------------------- 1.8/3.8 MB 1.5 MB/s eta 0:00:02
   ------------------- -------------------- 1.8/3.8 MB 1.5 MB/s eta 0:00:02
   --------------------- ------------------ 2.1/3.8 MB 1.4 MB/s eta 0:00:02
   --------------------------- ------------ 2.6/3.8 MB 1.5 MB/s eta 0:00:01
   ------------------------------ --------- 2.9/3.8 MB 1.5 MB/s eta 0:00:01
   -------------------------------- ------- 3.1/3.8 MB 1.5 MB/s eta 0:00:01
   -------------------------------------- - 3.7/3.8 MB 1.5 MB/s eta 0:00:01
   ----------------------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\UnItEd\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# 2) Connect to MySQL  --> change PASSWORD to your MySQL root password
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
import pandas as pd
import re

USER = "root"
PASSWORD = "your_password"
HOST = "localhost"
PORT = 3306
DB = "customer_churn"

pw = quote_plus(PASSWORD)

root_engine = create_engine(f"mysql+pymysql://{USER}:{pw}@{HOST}:{PORT}")
with root_engine.begin() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB}"))

engine = create_engine(f"mysql+pymysql://{USER}:{pw}@{HOST}:{PORT}/{DB}")
print("Connected to MySQL, database:", DB)

Connected to MySQL, database: customer_churn


In [3]:
# 3) Load clean_churn.csv into the table `telco` (replaces it if it already exists)
df = pd.read_csv("clean_churn.csv")
df.to_sql("telco", engine, if_exists="replace", index=False)
print(len(df), "rows loaded (expected 7043)")

7043 rows loaded (expected 7043)


In [4]:
# 4) Quick check
pd.read_sql(text("SELECT COUNT(*) AS rows_in_table FROM telco"), engine)

,rows_in_table
0,7043


In [5]:
# 5) Run all queries Q0 to Q15 from customer_analysis.sql and show each result
sql = open("customer_analysis.sql", encoding="utf-8").read()
sql = sql[sql.index("-- Q0"):]
blocks = re.split(r"(?=^-- Q\d+:)", sql, flags=re.M)

for block in blocks:
    block = block.strip()
    if not block:
        continue
    title = block.splitlines()[0]
    query = "\n".join(l for l in block.splitlines() if not l.strip().startswith("--"))
    print("=" * 70)
    print(title)
    display(pd.read_sql(text(query), engine))

-- Q0: Preview the data


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,No,Yes,No,1,No,No,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,No,No,No,45,No,No,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
5,9305-CDSKC,Female,No,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
6,1452-KIOVK,Male,No,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,0
7,6713-OKOMC,Female,No,No,No,10,No,No,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.90,0
8,7892-POOKP,Female,No,Yes,No,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
9,6388-TABGU,Male,No,No,Yes,62,Yes,No,DSL,Yes,...,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,0


-- Q1: Data quality check (rows vs unique customers)


,total_rows,unique_customers
0,7043,7043


-- Q2: Overall churn rate (baseline for every other result)


...

,contract,internet_service,customers,churn_rate,risk_rank
0,Month-to-month,Fiber optic,2128,54.61,1
1,Month-to-month,DSL,1223,32.22,2
2,One year,Fiber optic,539,19.29,3
3,Month-to-month,No,524,18.89,4
4,One year,DSL,570,9.30,5
5,Two year,Fiber optic,429,7.23,6
6,One year,No,364,2.47,7
7,Two year,DSL,628,1.91,8
8,Two year,No,638,0.78,9


-- Q14: Active high-risk customers (rule-based list)


,customer_id,tenure,contract,internet_service,monthly_charges
0,5760-IFJOZ,3,Month-to-month,Fiber optic,107.95
1,7145-FEJWU,12,Month-to-month,Fiber optic,105.30
2,6734-GMPVK,5,Month-to-month,Fiber optic,105.30
3,9547-ITEFG,9,Month-to-month,Fiber optic,102.60
4,7379-FNIUJ,2,Month-to-month,Fiber optic,100.20
5,4445-ZJNMU,9,Month-to-month,Fiber optic,99.30
6,6645-MXQJT,2,Month-to-month,Fiber optic,97.10
7,4566-NECEV,5,Month-to-month,Fiber optic,96.55
8,4060-LDNLU,7,Month-to-month,Fiber optic,96.20
9,4929-XIHVW,2,Month-to-month,Fiber optic,95.50


-- Q15: How many active customers match the Q14 rules (Q14 shows only 50)


,high_risk_active_customers
0,241
